# log-samples-eval-callback composite — cx22: log-samples eval callback then zero_grad(set_to_none=True) to resume training

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `log-samples-eval-callback`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "log-samples-eval-callback"
DD_ATOM_IDS = ["log-samples-eval-callback", "zero-grad-set-none"]
DD_SUBTOPICS = ["Logging: log-samples eval callback", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

After a generator/decoder eval pass produces sample images, the trainer has to **reset the gradients** before the next backward — otherwise the next step's `.backward()` accumulates on top of the LAST training step's `.grad` (PyTorch's `.grad` accumulator is an *add*, not an overwrite).

**The two atoms.**
- **log-samples-eval-callback** — periodic sample generation; does NOT touch parameter grads (it should be wrapped in `no_grad` / `inference_mode` to avoid pollution, but even if not, no `.backward()` is called inside).
- **zero-grad-set-none** — set each `p.grad = None` (NOT `p.grad.zero_()`). `set_to_none` is the modern PyTorch default because it skips the zero-fill and lets autograd allocate fresh storage next backward. Reads of `.grad` after `set_to_none` return `None`, not a zero tensor.

**Why compose them.** If your trainer fires the eval callback INSIDE the same step where it forgets to call zero_grad, the NEXT training step sees old `.grad` + new `.grad`. With `set_to_none=True`, the contract is even stricter: after the call, `.grad` IS `None` and the next backward writes a fresh tensor.

**Anatomy.**
```python
if step % log_every == 0:
    samples = sample_fn()                   # log-samples-eval-callback.
    sink[step] = samples
for p in params:                            # zero-grad-set-none.
    p.grad = None
```

### Composite Exercise — log-samples eval callback then zero_grad(set_to_none=True) to resume training

**Atoms exercised together**: `log-samples-eval-callback`, `zero-grad-set-none`

Implement `cx22_eval_then_zero_grad(params, step, log_every, sample_fn, sink)`.

Inputs:
- `params` — list of `t.Tensor` leaves with `requires_grad=True`. They may have `.grad` populated from a prior step.
- `step` — int.
- `log_every` — int.
- `sample_fn` — callable `() -> Any`.
- `sink` — list; append `(step, sample)` to it when the callback fires.

Required behaviour:
1. If `step % log_every == 0`, call `sample_fn()` and append `(step, sample)` to `sink` (atom: log-samples-eval-callback).
2. UNCONDITIONALLY (every call), for each `p in params`, set `p.grad = None` (atom: zero-grad-set-none). `None`, not a zero tensor.
3. Return `None`.

Test checks:
- After the call, every `p.grad is None` — even if it was populated before.
- Callback fires iff `step % log_every == 0`.
- A subsequent backward+grad-population works (proves `set_to_none=True` semantics don't break autograd plumbing — autograd will allocate a fresh `.grad` tensor).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx22_eval_then_zero_grad(params, step, log_every, sample_fn, sink):
    """Maybe fire eval callback, then set every p.grad = None. Returns None."""
    raise NotImplementedError

def _test_cx22():
    # Case A: pre-populated grads are wiped to None.
    p1 = t.randn(3, requires_grad=True)
    p2 = t.randn(2, 4, requires_grad=True)
    p1.grad = t.ones_like(p1)
    p2.grad = t.full_like(p2, 7.0)
    sink_a = []
    ret = cx22_eval_then_zero_grad([p1, p2], step=4, log_every=2, sample_fn=lambda: 'x', sink=sink_a)
    assert ret is None
    assert p1.grad is None, f'p1.grad should be None after set_to_none semantics; got {p1.grad}'
    assert p2.grad is None, f'p2.grad should be None; got {p2.grad}'
    assert sink_a == [(4, 'x')], f'callback should have fired with (4, "x"); got {sink_a}'

    # Case B: callback does NOT fire when step % log_every != 0, but grad is STILL zeroed.
    p3 = t.tensor([1.0, 2.0], requires_grad=True)
    p3.grad = t.tensor([99.0, 99.0])
    sink_b = []
    calls_b = {'n': 0}
    def _sb():
        calls_b['n'] += 1
        return 'shouldnt-fire'
    cx22_eval_then_zero_grad([p3], step=5, log_every=4, sample_fn=_sb, sink=sink_b)
    assert p3.grad is None, 'grad must be wiped unconditionally — even when callback skipped'
    assert sink_b == [], 'callback should not fire when step % log_every != 0'
    assert calls_b['n'] == 0

    # Case C: subsequent backward populates fresh grad — autograd plumbing not broken.
    p4 = t.tensor([2.0, 3.0], requires_grad=True)
    p4.grad = t.ones(2)
    cx22_eval_then_zero_grad([p4], step=0, log_every=1, sample_fn=lambda: 'samp', sink=[])
    assert p4.grad is None
    # Now do a fresh backward and confirm grad gets populated correctly.
    (p4.sum()).backward()
    assert p4.grad is not None, 'after zero_grad(set_to_none=True) + new backward, grad should be populated'
    assert t.allclose(p4.grad, t.ones(2)), f'd(sum(p))/dp = 1; got {p4.grad.tolist()}'

    # Case D: params with grad=None to begin with stay None (no AttributeError).
    p5 = t.randn(4, requires_grad=True)
    assert p5.grad is None
    cx22_eval_then_zero_grad([p5], step=2, log_every=1, sample_fn=lambda: None, sink=[])
    assert p5.grad is None

    # Case E: empty param list runs fine.
    sink_e = []
    cx22_eval_then_zero_grad([], step=3, log_every=3, sample_fn=lambda: 'fire', sink=sink_e)
    assert sink_e == [(3, 'fire')], 'callback should still fire for empty params'

    # Case F: callback fires on step 0 too (0 % log_every == 0).
    p6 = t.randn(2, requires_grad=True)
    p6.grad = t.zeros(2)
    sink_f = []
    cx22_eval_then_zero_grad([p6], step=0, log_every=5, sample_fn=lambda: 'init', sink=sink_f)
    assert sink_f == [(0, 'init')], 'step 0 % 5 == 0, callback should fire'
    assert p6.grad is None
    _dd_passed.add('cx22')

_test_cx22()

<details><summary>Show solution — cx22</summary>

```python
def cx22_eval_then_zero_grad(params, step, log_every, sample_fn, sink):
    # Atom A (log-samples-eval-callback): conditional sample generation.
    if step % log_every == 0:
        sample = sample_fn()
        sink.append((step, sample))
    # Atom B (zero-grad-set-none): unconditional grad reset, set_to_none=True semantics.
    for p in params:
        p.grad = None
```

The grad reset MUST be unconditional — even on steps where the eval callback doesn't fire, the trainer still finished a backward and needs a clean slate before the next one. Pairing the conditional eval with the unconditional zero_grad is the most common shape in real ARENA trainers. Note: setting `.grad = None` is preferred over `.grad.zero_()` because (a) it avoids a memset, and (b) it makes 'no backward happened yet' distinguishable from 'a backward happened with all-zero gradients'.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx22'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx22',
        'subtopics': ["Logging: log-samples eval callback", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()